# Week 3 — The Cleaning Sprint (Part 2: Finishing the Job)
**Syllabus mapping: Unit 2**

This notebook **continues from your teammate's cleaning notebook** — it does not redo her work.
She already handled missing values (median fill on the 6 affected columns). This notebook
picks up from her output (`Global_Warming_Cleaned.csv`) and finishes the remaining Stage 1
requirements from the rule book:

1. Upload her cleaned dataset
2. **Data Quality Report** (before further cleaning) — required by the rule book
3. Remove duplicate rows (she only checked for these)
4. Strip whitespace from `Country` (she only checked for this)
5. **Z-score outlier detection + removal** (the rule book asks for Z-score specifically; her notebook only used IQR to detect, never removed anything)
6. Final Data Quality Report (after cleaning) — proof of improvement
7. Save the fully-cleaned dataset


## 1. Upload her cleaned dataset
Run this cell, then upload `Global_Warming_Cleaned.csv` (the file she produced and downloaded).

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

## 2. Data Quality Report — BEFORE

The rule book requires code that *"automatically generates a Data Quality Report summarizing
the health of the dataset before analysis."* This function builds one report we can reuse
before and after cleaning, so you can show improvement.

In [ ]:
def data_quality_report(data, title="Data Quality Report"):
    print("=" * 60)
    print(title)
    print("=" * 60)
    print(f"Shape: {data.shape[0]} rows x {data.shape[1]} columns")
    print(f"Duplicate rows: {data.duplicated().sum()}")

    missing = data.isnull().sum()
    missing_pct = (missing / len(data)) * 100
    missing_report = pd.DataFrame({
        "Missing_Values": missing,
        "Missing_Percentage": missing_pct.round(2)
    })
    missing_report = missing_report[missing_report["Missing_Values"] > 0]
    print(f"\nColumns with missing values: {len(missing_report)}")
    if len(missing_report) > 0:
        print(missing_report)

    num_cols = data.select_dtypes(include=[np.number]).columns
    exclude = [c for c in num_cols if c.lower() in ("year", "id")]
    outlier_cols = [c for c in num_cols if c not in exclude]

    outlier_counts = {}
    for col in outlier_cols:
        z = np.abs(stats.zscore(data[col].dropna()))
        outlier_counts[col] = int((z > 3).sum())

    outlier_summary = pd.Series(outlier_counts).sort_values(ascending=False)
    outlier_summary = outlier_summary[outlier_summary > 0]
    print(f"\nColumns with Z-score outliers (|Z|>3): {len(outlier_summary)}")
    if len(outlier_summary) > 0:
        print(outlier_summary)

    if "Country" in data.columns:
        ws = (data["Country"] != data["Country"].str.strip()).sum()
        print(f"\nCountry values with leading/trailing whitespace: {ws}")

    print("=" * 60)
    return {
        "shape": data.shape,
        "duplicates": data.duplicated().sum(),
        "missing_total": int(missing.sum()),
        "outlier_total": int(outlier_summary.sum()) if len(outlier_summary) > 0 else 0
    }

before_report = data_quality_report(df, title="Data Quality Report — BEFORE further cleaning")

## 3. Remove duplicate rows

Her notebook checked for duplicates but didn't drop them. Finishing that step now.

In [ ]:
dup_count = df.duplicated().sum()
print("Duplicate rows found:", dup_count)

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

## 4. Strip whitespace from Country

Same story — she checked for this, didn't fix it.

In [ ]:
if "Country" in df.columns:
    ws_before = (df["Country"] != df["Country"].str.strip()).sum()
    df["Country"] = df["Country"].str.strip()
    ws_after = (df["Country"] != df["Country"].str.strip()).sum()
    print(f"Whitespace-padded Country values: {ws_before} -> {ws_after}")

## 5. Outlier detection & removal — Z-score

Her notebook used **IQR** to *detect* outliers but never removed them. The rule book calls
for **Z-score** as an accepted method alongside IQR — so this section adds that missing
piece and actually treats the outliers this time.

$$Z = \frac{x - \mu}{\sigma}$$

Rows with **|Z| > 3** on any numeric column are removed.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = [c for c in num_cols if c.lower() in ("year", "id")]
outlier_cols = [c for c in num_cols if c not in exclude_cols]

# Visual check first — boxplots
n_cols = 3
n_rows = int(np.ceil(len(outlier_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="skyblue")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("")

for j in range(len(outlier_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
z_scores = df[outlier_cols].apply(lambda x: np.abs(stats.zscore(x)))

outlier_counts = (z_scores > 3).sum().sort_values(ascending=False)
outlier_summary = pd.DataFrame({
    "Outlier_Count": outlier_counts,
    "Outlier_Percentage": (outlier_counts / len(df) * 100).round(3)
})
outlier_summary = outlier_summary[outlier_summary["Outlier_Count"] > 0]
print(outlier_summary)

outlier_mask = (z_scores > 3).any(axis=1)
print(f"\nTotal rows flagged as outliers (Z>3 on any column): {outlier_mask.sum()}")
print(f"Percentage of dataset: {round(outlier_mask.sum() / len(df) * 100, 2)}%")

In [ ]:
# Remove the flagged outlier rows
df_before_outlier_removal = df.shape[0]
df = df[~outlier_mask].reset_index(drop=True)

print(f"Rows before outlier removal: {df_before_outlier_removal}")
print(f"Rows after outlier removal: {df.shape[0]}")
print(f"Rows removed: {df_before_outlier_removal - df.shape[0]}")

## 6. Data Quality Report — AFTER

Same function, run again — this is the "before vs after" proof the rule book wants.

In [ ]:
after_report = data_quality_report(df, title="Data Quality Report — AFTER cleaning")

print("\nSummary of improvement:")
print(f"  Duplicates:  {before_report['duplicates']} -> {after_report['duplicates']}")
print(f"  Missing:     {before_report['missing_total']} -> {after_report['missing_total']}")
print(f"  Outliers:    {before_report['outlier_total']} -> {after_report['outlier_total']}")
print(f"  Shape:       {before_report['shape']} -> {after_report['shape']}")

## 7. Save the fully-cleaned dataset

This version has: her median-filled missing values, duplicates removed, whitespace stripped,
and outliers removed via Z-score. This is what should feed into the Week 4 EDA notebook.

In [ ]:
df.to_csv("Global_Warming_Fully_Cleaned.csv", index=False)
print("Saved: Global_Warming_Fully_Cleaned.csv")
print("Final shape:", df.shape)

# Uncomment to download directly:
# from google.colab import files
# files.download("Global_Warming_Fully_Cleaned.csv")